In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from sqlalchemy import text
import oracledb
import sys

In [2]:
#CONEXIONES DESTINO

DB_USER=config("USER2_BDI_DW")
DB_PASSWORD=config("PASS2_BDI_DW")
DB_NAME="GCSPE"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_DW")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

In [3]:
#CONEXIONES ORIGEN
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = 'USR_GSIT_GCHIARELLA'
pw = 'Ujpd9jQw#ZhCq3L0Y7'
hostname='10.0.0.49'
service_name="sas"
port = 1521

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

In [4]:
fec_ini = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_proceso=1", con=connection1)
fec_fin = pd.read_sql_query("SELECT fec_ter FROM etl_act where id_proceso=1", con=connection1)
fec_ini= fec_ini.iloc[0, 0]
fec_fin= fec_fin.iloc[0, 0]

In [5]:
# MAESTRO_DEPENDENCIAS = "SELECT * FROM GAAA.MCH_DEPENDENCIAS"
# MAESTRO_PERSONAS = "SELECT p.IDE_NUMERICO_PERSONA, p.COD_EDOCUMENT_PERSONA, p.NUM_DOCUMENT_PERSONA FROM GAAA.CSAMPERSON p"
# MAESTRO_USUARIOS = f"""
#     SELECT U.IDE_USUARIO, U.COD_USUARIO, FB.NOMBRE_APELLIDO, U.COD_EDOCUMENT_PERSONA, FB.PERFIL, FB.COD_DEPEN, FB.USUARIO_SIA FROM 
#         GAAA.SEGM_USUARIO U
#     LEFT OUTER JOIN GAAA.MCH_SEGM_USUARIO_FRONT_BACK FB
#         ON U.COD_USUARIO = FB.USUARIO_SAS
# """
# MAESTRO_EMPRESAS = "SELECT IDE_NUMERICO_ENTIDAD, NUM_DOCUMENT_ENTIDAD, TXT_RASOCIAL_ENTIDAD, TXT_MARCOMER_ENTIDAD FROM GAAA.CSAMENTIDA"

TRANSACCIONES = f"""
    SELECT
        'SAS' AS ORIGEN,
        'INSCRIPCION' AS ORIGEN2,
        CASE
            WHEN P.COD_ETIPO_OPERACIO = '100000' THEN 'ALTAS'
            WHEN P.COD_ETIPO_OPERACIO = '200000' THEN 'BAJAS'
            WHEN P.COD_ETIPO_OPERACIO = '300000' THEN 'CAMBIOS'
        END AS TRANSACCION,
        TO_DATE(P.FEC_REGISTRO_SISTEMA || ' ' || P.HOR_REGISTRO_SISTEMA, 'YYYYMMDD HH24:MI:SS') AS FECHA_HORA_REGISTRO,
        I.IDE_NUMERICO_PERSONA,
        per.COD_EDOCUMENT_PERSONA,
        per.NUM_DOCUMENT_PERSONA,
        P.ID_DEPENDEN_REGISTRA,
        P.IDE_USUARIO_REGISTRA,
        I.IDE_NUMERICO_ENTIDAD,
        I.IDE_NUMERICO_TITULAR,
        tit.COD_EDOCUMENT_PERSONA AS COD_EDOCUMENT_TITULAR,
        tit.NUM_DOCUMENT_PERSONA AS NUM_DOCUMENT_TITULAR,
        E.NUM_DOCUMENT_ENTIDAD
    FROM GAAA.CSATACCINS P
    INNER JOIN GAAA.CSATINSCRI I
        ON P.IDE_NUMERICO_INSCRIP = I.IDE_NUMERICO_INSCRIP
        AND P.COD_ETIPO_OPERACIO = I.COD_ETIPO_OPERACIO
        AND TO_CHAR(P.IDE_USUARIO_REGISTRA) = I.COD_USUARIO_SISTEMA
    LEFT JOIN GAAA.CSAMENTIDA E
        ON E.IDE_NUMERICO_ENTIDAD = I.IDE_NUMERICO_ENTIDAD
    LEFT JOIN GAAA.CSAMPERSON per
        ON I.IDE_NUMERICO_PERSONA = per.IDE_NUMERICO_PERSONA
    LEFT JOIN GAAA.CSAMPERSON tit
        ON I.IDE_NUMERICO_TITULAR = tit.IDE_NUMERICO_PERSONA
    WHERE P.COD_ETIPO_OPERACIO IN ('100000', '200000', '300000')
        AND P.COD_ECLASIFIC_COBERTUR IN ('01','02','04')
        AND P.COD_EVINCULO_PERSONA IN ('0101', '0102', '0103', '0104', '0105', '0106')

    UNION ALL

    SELECT
        'SAS' AS ORIGEN,
        'INSCRIPCION' AS ORIGEN2,
        CASE
            WHEN P.COD_ETIPO_OPERACIO = '100000' THEN 'ALTAS'
            WHEN P.COD_ETIPO_OPERACIO = '200000' THEN 'BAJAS'
            WHEN P.COD_ETIPO_OPERACIO = '300000' THEN 'CAMBIOS'
        END AS TRANSACCION,
        TO_DATE(P.FEC_REGISTRO_SISTEMA || ' ' || P.HOR_REGISTRO_SISTEMA, 'YYYYMMDD HH24:MI:SS') AS FECHA_HORA_REGISTRO,
        I.IDE_NUMERICO_PERSONA,
        per.COD_EDOCUMENT_PERSONA,
        per.NUM_DOCUMENT_PERSONA,
        P.ID_DEPENDEN_REGISTRA,
        P.IDE_USUARIO_REGISTRA,
        I.IDE_NUMERICO_ENTIDAD,
        I.IDE_NUMERICO_TITULAR,
        tit.COD_EDOCUMENT_PERSONA AS COD_EDOCUMENT_TITULAR,
        tit.NUM_DOCUMENT_PERSONA AS NUM_DOCUMENT_TITULAR,
        E.NUM_DOCUMENT_ENTIDAD
    FROM GAAA.CSATACCINS P
    INNER JOIN GAAA.CSAHINSCRI I
        ON P.IDE_NUMERICO_INSCRIP = I.IDE_NUMERICO_INSCRIP
        AND P.COD_ETIPO_OPERACIO = I.COD_ETIPO_OPERACIO
        AND TO_CHAR(P.IDE_USUARIO_REGISTRA) = I.COD_USUARIO_SISTEMA
    LEFT JOIN GAAA.CSAMENTIDA E
        ON E.IDE_NUMERICO_ENTIDAD = I.IDE_NUMERICO_ENTIDAD
    LEFT JOIN GAAA.CSAMPERSON per
        ON I.IDE_NUMERICO_PERSONA = per.IDE_NUMERICO_PERSONA
    LEFT JOIN GAAA.CSAMPERSON tit
        ON I.IDE_NUMERICO_TITULAR = tit.IDE_NUMERICO_PERSONA
    WHERE P.COD_ETIPO_OPERACIO IN ('100000', '200000', '300000')
        AND P.COD_ECLASIFIC_COBERTUR IN ('01','02','04')
        AND P.COD_EVINCULO_PERSONA IN ('0101', '0102', '0103', '0104', '0105', '0106')
"""

In [6]:
# Función para dividir las fechas en bloques mensuales
def generate_monthly_blocks(start_date, end_date):
    blocks = []
    current_start = start_date

    while current_start < end_date:
        next_start = (current_start + timedelta(days=31)).replace(day=1)
        blocks.append((current_start, min(next_start, end_date)))
        current_start = next_start

    return blocks

# Bloques mensuales
monthly_blocks = generate_monthly_blocks(fec_ini, fec_fin)

# Función para cargar datos en bloques
def load_master_data_in_chunks(nombre, query, source_conn, target_conn, chunk_size=250000):
    offset = 0
    while True:
        chunk_query = f"""
            SELECT * FROM (
                SELECT A.*, ROWNUM rnum FROM (
                    {query}
                ) A WHERE ROWNUM <= {offset + chunk_size}
            ) WHERE rnum > {offset}
        """
        chunk = pd.read_sql_query(chunk_query, con=source_conn)
        if chunk.empty:
            break
        chunk = chunk.drop(columns=['rnum'])  # Eliminar la columna rnum
        chunk.to_sql(nombre, con=target_conn, if_exists='append', index=False)
        offset += chunk_size
        print(f"Processed {offset} rows from {nombre}")

# Borrado de los registros en la BD de destino para el rango de fechas correspondiente
# delete_query = f"""
#     DELETE FROM public."TRANSACCIONES_OSPES_SEGUROS" t
#     WHERE FECHA_HORA_REGISTRO >= TO_DATE('{fec_ini.strftime('%Y-%m-%d')}', 'YYYY-MM-DD')
# """
# connection1.execute(delete_query)
# print(f'Borrado de datos existente para el rango FEC_INI: {fec_ini}')

# Lectura y carga de las transacciones en bloques mensuales
for start_date, end_date in monthly_blocks:
    print(f'PROCESANDO: TRANSACCIONES - FEC_INI:{start_date}')
    TRANSACCIONES_BLOCK = f"""
        SELECT * FROM (
            {TRANSACCIONES}
        ) WHERE FECHA_HORA_REGISTRO >= TO_DATE('{start_date.strftime('%Y-%m-%d')}', 'YYYY-MM-DD')
            AND FECHA_HORA_REGISTRO < TO_DATE('{end_date.strftime('%Y-%m-%d')}', 'YYYY-MM-DD')
    """
    transacciones_df = pd.read_sql_query(TRANSACCIONES_BLOCK, con=connection0)
    transacciones_df.to_sql('TRANSACCIONES_OSPES_SEGUROS', con=connection1, if_exists='append', index=False)
    print(f'FINALIZADO: TRANSACCIONES - FEC_INI:{start_date}')

# Cargar datos de los maestros en bloques
# load_master_data_in_chunks('MAESTRO_DEPENDENCIAS', MAESTRO_DEPENDENCIAS, connection0, connection1)
# load_master_data_in_chunks('MAESTRO_PERSONAS', MAESTRO_PERSONAS, connection0, connection1)
# load_master_data_in_chunks('MAESTRO_USUARIOS', MAESTRO_USUARIOS, connection0, connection1)
# load_master_data_in_chunks('MAESTRO_EMPRESAS', MAESTRO_EMPRESAS, connection0, connection1)

# Cerrar conexiones
now2 = datetime.now().strftime('%Y-%m-%d')
query1=f"UPDATE etl_act SET fec_ini ='{now2}' WHERE id_proceso=1"
query2=f"UPDATE etl_act SET fec_act ='{now2}' WHERE id_proceso=1"
c1= text(query1)
c2= text(query2)
connection1.execute(c1)
connection1.execute(c2)
connection0.close()
connection1.close()

PROCESANDO: TRANSACCIONES - FEC_INI:2024-08-29
FINALIZADO: TRANSACCIONES - FEC_INI:2024-08-29
PROCESANDO: TRANSACCIONES - FEC_INI:2024-09-01
FINALIZADO: TRANSACCIONES - FEC_INI:2024-09-01
PROCESANDO: TRANSACCIONES - FEC_INI:2024-10-01
FINALIZADO: TRANSACCIONES - FEC_INI:2024-10-01
PROCESANDO: TRANSACCIONES - FEC_INI:2024-11-01
FINALIZADO: TRANSACCIONES - FEC_INI:2024-11-01
PROCESANDO: TRANSACCIONES - FEC_INI:2024-12-01
FINALIZADO: TRANSACCIONES - FEC_INI:2024-12-01
